### Neural Style Transfer

**Neural Transer Recipe**

Using a content image (e.g. cat photo) and a style image (e.g. Starry Night by Van Gogh) that the user will upload, this method will give the user a combination via neural style transfer.

Uses the pre-trained VGG19 CNN to extract multi-layer features. Content representation is extracted from deeper layers (structure and shapes) and style representation is extracted from shallower layers (texture and colour). Algorithm loop optimises pixels of the generated image to minimise the total loss (calculated using the loss function). 

If gpu is available use a larger image size and use loader to perform image transformations i.e. pre-processing for vgg19, used to load the image (also adds a batch dimension [1, 3, H, W]). Use the unloader to reverse normalization to convert tensors back to viewable images, used to save the final stylised image. 

The gram matrix measures feature correlations within a layer to capture style, hence encodes texture and colour. N.B. Two images have similar styles if their gram matrices are similar.

Perform feature extraction by running the vgg19 network which ectracts intermediate activations i.e. feature maps at specific layers. Content layer is conv4_2 to caapture structure and style layers conv1_1, conv2_1, conv3_1, conv4_1, conv5_1 to capture texture.

Total loss function is calculated using content and style loss i.e. measures how close generated features are to content features and how close generated Gram matrices are to style Gram matrices, respectively. The total variation loss is used for smoothness and to remove noise.

Optimisation loop:
1. Load content and style images
2. Load pretrained VGG19
3. Extract features and Gram matrices
4. Initialize generated image, optimise pixels of this tensor directly
5. Define optimizer
6. Training loop for n steps which iteratively adjusts pixels of generated to minimize total loss
7. Save the final image

Parameters used in the style transfer like content_weight, style_weight, tv_weight can get more or less stylized outputs.

N.B. No epochs since no parameters are being trained and model is pretrained, since only generated image is being optimised. (Training the image not the model) Instead each optimisation step updates generated image tensor.

TO-DO: 
- try out more style focused approach apart from this version i.e. change parameter weights and optimiser


In [ ]:
import torch    
from torchvision import transforms
from PIL import Image
import torch.optim as optim
import torchvision.models as models
from torchvision.models import VGG19_Weights
import matplotlib.pyplot as plt
from pathlib import Path
import imageio
import numpy as np

cuda
2.11.0+cu130
torch.Size([1, 16, 222, 222])


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

image_size = 512 if torch.cuda.is_available() else 256  

normalization_mean = [0.485, 0.456, 0.406]
normalization_std = [0.229, 0.224, 0.225]
normalize = transforms.Normalize(mean=normalization_mean,
                                 std=normalization_std)

transform = transforms.Compose([
    transforms.Resize(image_size),              
    transforms.CenterCrop(image_size),          
    transforms.ToTensor(),
    normalize
])

unloader = transforms.Compose([
    transforms.Normalize(
        mean=[0, 0, 0],
        std=[1/0.229, 1/0.224, 1/0.225]),
    transforms.Normalize(
        mean=[-0.485, -0.456, -0.406],
        std=[1, 1, 1]),
    transforms.ToPILImage()
])

def load_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0)  # Add batch dimension
    return image.to(device) 

def tensor_to_pil(tensor):
    image = tensor.detach().cpu().clone().squeeze(0)
    image = unloader(image)  
    return image

def save_image(tensor, path):
    image = tensor.detach().cpu().clone().squeeze(0)
    image = image * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image = image + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    image = torch.clamp(image, 0, 1)
    transforms.ToPILImage()(image).save(path)

def save_gif(snapshots, path, duration=0.5):
    frames = [np.array(img) for img in snapshots]
    imageio.mimsave(path, frames, duration=duration)
    print(f"Saved GIF to {path}")

def gram_matrix(tensor):
    # tensor shape: (batch_size=1, channels, height, width)
    b, c, h, w = tensor.size()
    features = tensor.view(c, h * w)      # reshape to (channels, height*width)
    G = torch.mm(features, features.t())  # compute Gram matrix (channels x channels)
    return G / (2 * c * h * w)  

# Load VGG19 model with pre-trained weights 
vgg = models.vgg19(weights=VGG19_Weights.DEFAULT).features.to(device).eval()
for param in vgg.parameters():
    param.requires_grad = False

#content_layer = ['conv4_2']
#style_layers = ['conv_1', 'conv_2', 'conv_3', 'conv_4', 'conv_5']

content_layer = '21' 
style_layers = ['0', '5', '10', '19', '28']

def get_content_feature(image):
    x = image
    for i, layer in enumerate(vgg):
        x = layer(x)
        if str(i) == content_layer:
            return x
        
def get_style_features(image, model, layers):
    features = []
    x = image
    for i, layer in enumerate(model):
        x = layer(x)
        if str(i) in layers:
            features.append(x)
    return features

def compute_content_loss(generated_features, content_features):
    return torch.nn.functional.mse_loss(generated_features, content_features)

def compute_style_loss(generated_features, style_features):
    loss = 0
    for gen_feat, style_feat in zip(generated_features, style_features):
        gram_generated = gram_matrix(gen_feat)
        gram_style = gram_matrix(style_feat)
        loss += torch.nn.functional.mse_loss(gram_generated, gram_style)
    return loss

def run_style_transfer(content_img_path, style_img_path, output_path, num_steps=300,
                    init_from_content=True,
                    snapshot_interval=50,
                    init_snapshot_interval=5,
                    show_progress=True):
    
    # Load images 
    content_img = load_image(content_img_path)
    style_img = load_image(style_img_path)

    # Extract Content and Style features from both images, respectively 
    style_features = get_style_features(style_img, vgg, style_layers)
    content_features = get_content_feature(content_img)

    # Initialize generated image
    if init_from_content:
        generated_img = content_img.clone().to(device).requires_grad_(True)
    else:
        generated_img = torch.randn((1, 3, 512, 512), device=device, requires_grad=True)

    optimizer = optim.LBFGS([generated_img])

    # Loss tracking for graph plot
    losses = {"total": [], "content": [], "style": []}
    current_losses = {"total": None, "content": None, "style": None}
    snapshots = []       # sparse for plotting
    gif_frames = []      # dense for GIF
    init_snapshots = []
        
    def closure():
        optimizer.zero_grad()

        # Forward Pass
        gen_content = get_content_feature(generated_img)
        gen_style = get_style_features(generated_img, vgg, style_layers)

        # Compute style and content loss
        c_loss = compute_content_loss(gen_content, content_features)
        s_loss = compute_style_loss(gen_style, style_features)
        s_loss = (1e9*s_loss)

        total_loss = c_loss + s_loss
        total_loss.backward()

        current_losses["total"] = total_loss.item()
        current_losses["content"] = c_loss.item()
        current_losses["style"] = s_loss.item()

        return total_loss

    for step in range(num_steps):
        optimizer.step(closure)

        losses["total"].append(current_losses["total"])
        losses["content"].append(current_losses["content"])
        losses["style"].append(current_losses["style"])

        # GIF
        gif_frames.append(tensor_to_pil(generated_img.detach()))

        # Snapshot plots 
        if step % snapshot_interval == 0 or step == 299:
            snapshots.append(gif_frames[-1])

        if step % init_snapshot_interval == 0 and step<=25:
            init_snapshots.append(gif_frames[-1])

        if show_progress:
            print(f"Step {step}, Total Loss: {current_losses['total']:.4f}, "
            f"Content Loss: {current_losses['content']:.4f}, "
            f"Style Loss: {current_losses['style']:.4f}")
        
    # Save output 
    with torch.no_grad():
        final_img = generated_img.detach().clone()

    save_image(final_img, output_path)
    print(f"Saved stylised image to {output_path}")

    return output_path, gif_frames, snapshots, init_snapshots, snapshot_interval, init_snapshot_interval, losses

cuda


In [5]:
content_path=Path(r"ref_photos\comino-swimming-snorkeling-malta.jpg")
style_path=Path(r"ref_paintings\Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg")
output_path=Path(r"generated_images\styled_new_method_test.jpg")

output_path, gif_frames, snapshots, init_snapshots, snapshot_interval, init_snapshot_interval, losses = run_style_transfer(
    content_img_path=content_path,
    style_img_path=style_path, 
    output_path=output_path,
    num_steps=300,
    snapshot_interval=50,
    init_snapshot_interval=5,
    show_progress=True
)

FileNotFoundError: [Errno 2] No such file or directory: 'ref_photos\\comino-swimming-snorkeling-malta.jpg'

In [1]:
# Output loss plot
plt.figure(figsize=(10, 5))
steps = list(range(len(losses["total"])))
plt.plot(steps, losses["total"], label="Total Loss")
plt.plot(steps, losses["content"], label="Content Loss", alpha=0.7)
plt.plot(steps, losses["style"], label="Style Loss", alpha=0.7)
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Neural Style Transfer Loss")
plt.legend()
plt.grid(True)
plt.show()

# Plot intermediate snapshots
fig, axes = plt.subplots(1, len(snapshots), figsize=(15, 5))
if len(snapshots) == 1:
    axes = [axes]

plt.title(f"Snapshots at every {snapshot_interval} Steps ")
for i, img in enumerate(snapshots):
    axes[i].imshow(img)
    axes[i].set_title(f"Step {i * snapshot_interval}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

# Plot initial snapshots
plt.title("Initial Steps of Neural Style Transfer")
fig, axes = plt.subplots(1, len(init_snapshots), figsize=(15, 5))
if len(init_snapshots) == 1:
    axes = [axes]
for i, img in enumerate(init_snapshots):
    axes[i].imshow(img)
    axes[i].set_title(f"Step {i * init_snapshot_interval}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

# Create a gif of the neural style transfer
gif_path = output_path.with_suffix(".gif")
frames = [np.array(img) for img in gif_frames]
imageio.mimsave(gif_path, frames, duration=0.08)
print(f"Saved GIF to {gif_path}")

NameError: name 'plt' is not defined